In [ ]:
import json
import requests

URL = "https://gis-api.aiesec.org/graphql"

QUERY = """
query GetAllOpportunitiesQuery(
  $page: Int, $per_page: Int, $filters: OpportunityFilter,
  $cdn_region: String, $gpPage: Int, $gpPerPage: Int,
  $gpFilters: OpportunityFilter, $skipGpOpportunities: Boolean!,
  $cover_photo_size: String, $org_photo_size: String
) {
  allOpportunity: opportunities(
    filters: $filters
    pagination: {page: $page, per_page: $per_page}
  ) {
    data {
      ...OpportunityFragment
      __typename
    }
    paging {
      total_items
      total_pages
      current_page
      __typename
    }
    __typename
  }
  globalPartnersOpportunities: opportunities(
    filters: $gpFilters
    pagination: {page: $gpPage, per_page: $gpPerPage}
  ) @skip(if: $skipGpOpportunities) {
    data {
      ...OpportunityFragment
      __typename
    }
    paging {
      total_items
      total_pages
      current_page
      __typename
    }
    __typename
  }
}

fragment OpportunityFragment on Opportunity {
  __typename
  id
  title
  description
  project_name
  project_description
  location
  company_description
  is_favourited
  partner_type
  experience_type
  remote_opportunity
  applicants_count
  earliest_start_date
  duration
  opportunity_duration_type { __typename id duration_type }
  cover_photo(cdn_region: $cdn_region, size: $cover_photo_size)
  programme { __typename id short_name short_name_display }
  branch {
    __typename
    id
    address_detail { __typename id country }
    company {
      __typename
      id
      name
      profile_photo(cdn_region: $cdn_region, size: $org_photo_size)
      address_detail { __typename id country }
    }
  }
  host_lc {
    __typename
    id
    address_detail { __typename id country }
  }
  backgrounds { __typename id constant_id constant_name option }
  skills      { __typename id constant_id constant_name option }
  languages   { __typename id constant_id constant_name option }
  nationalities { __typename id constant_id constant_name option }
  study_levels { __typename id name option }
  remote_experience_duration
  remote_experience_salary
  specifics_info {
    __typename
    salary
    salary_periodicity
    salary_currency { __typename id alphabetic_code }
  }
  sdg_info {
    __typename
    id
    sdg_target {
      __typename
      id
      description
      goal_index
      target
      target_id
      parent {
        __typename
        id
        name
        short_name
        sdg_goal_hex_code
      }
    }
  }
  sub_product { __typename id name }
  organisation { __typename id name }
  weekly_activities(public_facing_only: true) {
    __typename
    id
    activity
    week
    is_active
  }
  role_info {
    __typename
    selection_process
    learning_points
    learning_points_list
  }
  work_hours
  available_slots {
    __typename
    id
    applications_close_date
    available_openings
    start_date
    end_date
    status
  }
  fee_and_health_insurance
  application_processing_time
  selection_processes(first: 50) {
    __typename
    edges {
      __typename
      node {
        __typename
        id
        title
        no_of_days
      }
    }
  }
  logistics_info {
    __typename
    accommodation_covered
    accommodation_provided
    computer_provided
    food_covered
    food_provided
    no_of_meals
    transportation_covered
    transportation_provided
  }
}
"""

VARIABLES = {
  "page": 1,
  "per_page": 1000,
  "filters": {
    "q": None,
    "sort": "created_at",
    "sort_direction": "desc",
    "programmes": [8],
    "earliest_start_date": {"from": "2025-11-08"},
    "exclude_home_mcs": [1559]
  },
  "cdn_region": "Global",
  "skipGpOpportunities": False,
  "cover_photo_size": "552x312",
  "org_photo_size": "300x300",
  "gpPage": 1,
  "gpPerPage": 50,
  "gpFilters": {
    "q": None,
    "sort": "created_at",
    "sort_direction": "desc",
    "programmes": [8],
    "earliest_start_date": {"from": "2025-11-08"},
    "exclude_home_mcs": [1559],
    "partner_type": ["global"]
  }
}

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"e316ebe109dd84ed16734e5161a2d236d0a7e6daf499941f7c110078e3c75493"
}

payload = {
    "query": QUERY,
    "variables": VARIABLES
}

response = requests.post(URL, json=payload, headers=HEADERS)

response.raise_for_status()
data = response.json()

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "data": {
    "allOpportunity": {
      "data": [
        {
          "__typename": "Opportunity",
          "id": "1329997",
          "title": "Recruitment Consultant",
          "description": "Step into a fast-paced sales environment where your drive directly shapes your success. As a Recruitment Consultant, you’ll build your own market, win new business, and connect top talent with industry-leading clients—all while earning uncapped commission and mastering a proven sales training programme. This is a role for ambitious, competitive individuals who thrive on targets, love relationship-building, and want a career where hard work truly pays off.",
          "project_name": "",
          "project_description": "",
          "location": "Prague, Czechia",
          "company_description": "NonStop is a leading recruitment agency, offering real career opportunities to ambitious and determined people. Founded in London in 2000, the company has expanded throughout Europe and the USA, 

In [ ]:
import json
import pandas as pd

# ---- 1. Load JSON file ----
# with open("/content/opportunities.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# ---- 2. Extract both lists ----
all_opps = data["data"]["allOpportunity"]["data"]
global_opps = data["data"]["globalPartnersOpportunities"]["data"]

# Combine all opportunity objects
all_data = all_opps + global_opps

# ---- 3. Flatten EVERYTHING ----
df = pd.json_normalize(
    all_data,
    sep="__"
)

# ---- 4. Show info ----
df.head()
df.columns.tolist()
len(df)

# ---- 5. Export full CSV with all fields ----
df.to_csv("opportunities_full_flat.csv", index=False)


In [ ]:
# Load the flattened CSV
df = pd.read_csv("opportunities_full_flat.csv")

# List of columns to keep (EXACT names you sent)
cols_to_keep = [
    "id",
    "title","description","location",
    "company_description","applicants_count",
    "backgrounds","skills","languages","nationalities","study_levels",
    "available_slots","application_processing_time",
    "opportunity_duration_type__duration_type","programme__id",
    "branch__company__name",
    "branch__company__address_detail__country",
    "host_lc__address_detail__country",
    "specifics_info__salary","specifics_info__salary_currency__alphabetic_code",
    "sub_product__name","organisation__name","role_info__learning_points",
    "fee_and_health_insurance__total","fee_and_health_insurance__currency",
    "fee_and_health_insurance__country","selection_processes__edges",
    "logistics_info__accommodation_covered","logistics_info__accommodation_provided",
    "logistics_info__computer_provided","logistics_info__food_covered",
    "logistics_info__food_provided","logistics_info__no_of_meals",
    "logistics_info__transportation_covered","logistics_info__transportation_provided"
]

# Keep only existing columns (skips any missing column safely)
existing_cols = [c for c in cols_to_keep if c in df.columns]
df = df[existing_cols].copy()
df.head()


,id,title,description,location,company_description,applicants_count,backgrounds,skills,languages,nationalities,...,fee_and_health_insurance__country,selection_processes__edges,logistics_info__accommodation_covered,logistics_info__accommodation_provided,logistics_info__computer_provided,logistics_info__food_covered,logistics_info__food_provided,logistics_info__no_of_meals,logistics_info__transportation_covered,logistics_info__transportation_provided
0,1329997,Recruitment Consultant,Step into a fast-paced sales environment where...,"Prague, Czechia","NonStop is a leading recruitment agency, offer...",1,"[{'__typename': 'ConstantMap', 'id': '17627465...","[{'__typename': 'ConstantMap', 'id': '17627466...","[{'__typename': 'ConstantMap', 'id': '17627466...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,provided,provided,covered,not_provided,1,not_covered,not_provided
1,1329970,Logistics Coordinator Intern,Hilti is where innovation is improving product...,"Panamá, Provincia de Panamá, Panamá","We stand for quality, innovation and direct cu...",3,"[{'__typename': 'ConstantMap', 'id': '17626746...","[{'__typename': 'ConstantMap', 'id': '17626747...","[{'__typename': 'ConstantMap', 'id': '17626746...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,not_provided,provided,not_covered,not_provided,0,not_covered,not_provided
2,1329930,Sx Marketing assistant LATAM GEM,Provide administrative support to ensure smoot...,"Panamá, Provincia de Panamá, Panamá",Alcon is a Swiss–American medical company spec...,8,"[{'__typename': 'ConstantMap', 'id': '17625721...","[{'__typename': 'ConstantMap', 'id': '17625721...","[{'__typename': 'ConstantMap', 'id': '17625721...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,not_provided,provided,not_covered,not_provided,0,not_covered,not_provided
3,1329918,"Accelerate Romania | Marketing, Social Media &...",- GoodStory is an events and marketing company...,"Bucharest, Romania",GoodStory thus means a success story in three ...,7,"[{'__typename': 'ConstantMap', 'id': '17625393...","[{'__typename': 'ConstantMap', 'id': '17625394...","[{'__typename': 'ConstantMap', 'id': '17625394...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",covered,provided,provided,not_covered,not_provided,0,not_covered,not_provided
4,1329895,Office Manager (EU only),Caps & Collars GmbH is looking for a fluent En...,"Hamburg, Deutschland",Caps & Collars is a German innovation boutique...,30,"[{'__typename': 'ConstantMap', 'id': '17625387...","[{'__typename': 'ConstantMap', 'id': '17625388...","[{'__typename': 'ConstantMap', 'id': '17625387...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,not_provided,not_provided,not_covered,not_provided,0,not_covered,not_provided


In [ ]:
df = df.rename(columns={
    "company_description": "company_desc",
    "applicants_count": "applicants",
    "application_processing_time": "processing_time",
    "opportunity_duration_type__duration_type": "duration_type",
    "programme__id": "programme_id",
    "branch__company__name": "company_name",
    "branch__company__address_detail__country": "company_country",
    "host_lc__address_detail__country": "host_country",
    "specifics_info__salary": "salary",
    "specifics_info__salary_currency__alphabetic_code": "salary_currency",
    "sub_product__name": "sub_product",
    "organisation__name": "organisation",
    "role_info__learning_points": "learning_points",
    "fee_and_health_insurance__total": "total_fee",
    "fee_and_health_insurance__currency": "fee_currency",
    "fee_and_health_insurance__country": "fee_country",
    "logistics_info__accommodation_covered": "accom_covered",
    "logistics_info__accommodation_provided": "accom_provided",
    "logistics_info__computer_provided": "computer",
    "logistics_info__food_covered": "food_covered",
    "logistics_info__food_provided": "food_provided",
    "logistics_info__no_of_meals": "meals",
    "logistics_info__transportation_covered": "transport_covered",
    "logistics_info__transportation_provided": "transport_provided"
})
df.columns.tolist()

['id',
 'title',
 'description',
 'location',
 'company_desc',
 'applicants',
 'backgrounds',
 'skills',
 'languages',
 'nationalities',
 'study_levels',
 'available_slots',
 'processing_time',
 'duration_type',
 'programme_id',
 'company_name',
 'company_country',
 'host_country',
 'salary',
 'salary_currency',
 'sub_product',
 'organisation',
 'learning_points',
 'total_fee',
 'fee_currency',
 'fee_country',
 'selection_processes__edges',
 'accom_covered',
 'accom_provided',
 'computer',
 'food_covered',
 'food_provided',
 'meals',
 'transport_covered',
 'transport_provided']

In [ ]:
df.head()

,id,title,description,location,company_desc,applicants,backgrounds,skills,languages,nationalities,...,fee_country,selection_processes__edges,accom_covered,accom_provided,computer,food_covered,food_provided,meals,transport_covered,transport_provided
0,1329997,Recruitment Consultant,Step into a fast-paced sales environment where...,"Prague, Czechia","NonStop is a leading recruitment agency, offer...",1,"[{'__typename': 'ConstantMap', 'id': '17627465...","[{'__typename': 'ConstantMap', 'id': '17627466...","[{'__typename': 'ConstantMap', 'id': '17627466...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,provided,provided,covered,not_provided,1,not_covered,not_provided
1,1329970,Logistics Coordinator Intern,Hilti is where innovation is improving product...,"Panamá, Provincia de Panamá, Panamá","We stand for quality, innovation and direct cu...",3,"[{'__typename': 'ConstantMap', 'id': '17626746...","[{'__typename': 'ConstantMap', 'id': '17626747...","[{'__typename': 'ConstantMap', 'id': '17626746...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,not_provided,provided,not_covered,not_provided,0,not_covered,not_provided
2,1329930,Sx Marketing assistant LATAM GEM,Provide administrative support to ensure smoot...,"Panamá, Provincia de Panamá, Panamá",Alcon is a Swiss–American medical company spec...,8,"[{'__typename': 'ConstantMap', 'id': '17625721...","[{'__typename': 'ConstantMap', 'id': '17625721...","[{'__typename': 'ConstantMap', 'id': '17625721...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,not_provided,provided,not_covered,not_provided,0,not_covered,not_provided
3,1329918,"Accelerate Romania | Marketing, Social Media &...",- GoodStory is an events and marketing company...,"Bucharest, Romania",GoodStory thus means a success story in three ...,7,"[{'__typename': 'ConstantMap', 'id': '17625393...","[{'__typename': 'ConstantMap', 'id': '17625394...","[{'__typename': 'ConstantMap', 'id': '17625394...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",covered,provided,provided,not_covered,not_provided,0,not_covered,not_provided
4,1329895,Office Manager (EU only),Caps & Collars GmbH is looking for a fluent En...,"Hamburg, Deutschland",Caps & Collars is a German innovation boutique...,30,"[{'__typename': 'ConstantMap', 'id': '17625387...","[{'__typename': 'ConstantMap', 'id': '17625388...","[{'__typename': 'ConstantMap', 'id': '17625387...",[],...,United States,"[{'__typename': 'SelectionProcessEdge', 'node'...",not_covered,not_provided,not_provided,not_covered,not_provided,0,not_covered,not_provided


In [ ]:
df.shape

(374, 35)

In [ ]:
df.isnull().sum()

,0
id,0
title,0
description,0
location,1
company_desc,0
applicants,0
backgrounds,0
skills,0
languages,0
nationalities,0


In [ ]:
df.duplicated().sum()

np.int64(12)

In [ ]:
import ast #for converting str to list

def parse_constant_name_list(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['constant_name'])
    return L

In [ ]:
df['backgrounds'] = df['backgrounds'].apply(parse_constant_name_list)
df['backgrounds']

,backgrounds
0,"[Sales, Marketing, Sports]"
1,[Business administration]
2,[Marketing]
3,"[Marketing, Communication & journalism, Graphi..."
4,"[Business administration, Tourism & hotel mana..."
...,...
369,[Communication & journalism]
370,[Communication & journalism]
371,"[Computer engineering, Computer sciences, Soft..."
372,[Human Resources]


In [ ]:
df['skills'] = df['skills'].apply(parse_constant_name_list)
df['skills']


,skills
0,"[Communication, Writing, Negotiation, Database..."
1,"[SAP, Database Management, Database Administra..."
2,"[Microsoft Excel, Microsoft Powerpoint, Data A..."
3,"[Photoshop, Social Media Management, Microsoft..."
4,"[Event Planning, Communication, Client servici..."
...,...
369,"[Internet usage, Windows PC usage, Language Te..."
370,"[Internet usage, Windows PC usage, Language Te..."
371,"[AS/400, C, C#, C++, ASP, ERP, DNS, F#, iOS, I..."
372,"[Relationship Management, Data Analytics, Recr..."


In [ ]:
for col in ["languages", "nationalities"]:
    if col in df.columns:
        df[col] = df[col].apply(parse_constant_name_list)
    else:
        df[col] = [[]] * len(df)

In [ ]:
def normalize_study_levels(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

df['study_levels'] = df['study_levels'].apply(normalize_study_levels)

df['study_levels'].head()

,study_levels
0,[High School]
1,[Bachelor]
2,[Bachelor]
3,[Bachelor]
4,[Bachelor]


In [ ]:
import re

# Remove all HTML tags
TAG_RE = re.compile(r"<.*?>")

def strip_html(text: str) -> str:
    text = TAG_RE.sub(" ", str(text))
    return re.sub(r"\s+", " ", text).strip()

# Convert stringified list → python list
def to_list(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except:
        return [val]


In [ ]:
df["learning_points"] = (
    df["learning_points"]
      .fillna("")
      .apply(strip_html)
      .str.lower()
)
df["learning_points"].head()

,learning_points
0,build and develop a niche market by expanding ...
1,communication with logistics stakeholders: pla...
2,end-to-end management and tracking of promotio...
3,- assisting with photo and video content creat...
4,make sure the co-working space is clean and fu...


In [ ]:
# create a new new_df_opp from the cleaned cols
df_opp = df[['id','title','company_name','description','backgrounds','skills','languages','study_levels','learning_points']]
df_opp.head()

,id,title,company_name,description,backgrounds,skills,languages,study_levels,learning_points
0,1329997,Recruitment Consultant,Non Stop Consulting,Step into a fast-paced sales environment where...,"[Sales, Marketing, Sports]","[Communication, Writing, Negotiation, Database...","[Dutch, English, French]",[High School],build and develop a niche market by expanding ...
1,1329970,Logistics Coordinator Intern,HILTI Panama,Hilti is where innovation is improving product...,[Business administration],"[SAP, Database Management, Database Administra...","[Spanish, English]",[Bachelor],communication with logistics stakeholders: pla...
2,1329930,Sx Marketing assistant LATAM GEM,Alcon Centroamérica PA,Provide administrative support to ensure smoot...,[Marketing],"[Microsoft Excel, Microsoft Powerpoint, Data A...","[Spanish, English]",[Bachelor],end-to-end management and tracking of promotio...
3,1329918,"Accelerate Romania | Marketing, Social Media &...",goodStory,- GoodStory is an events and marketing company...,"[Marketing, Communication & journalism, Graphi...","[Photoshop, Social Media Management, Microsoft...",[English],[Bachelor],- assisting with photo and video content creat...
4,1329895,Office Manager (EU only),Caps & Collars GmbH,Caps & Collars GmbH is looking for a fluent En...,"[Business administration, Tourism & hotel mana...","[Event Planning, Communication, Client servici...",[English],[Bachelor],make sure the co-working space is clean and fu...


In [ ]:
def clean_text(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9]+", " ", t)
    return t.strip()


#split description abd learning points
df_opp['description'] = df_opp['description'].apply(lambda x: x.split())
df_opp['learning_points'] = df_opp['learning_points'].apply(lambda x: x.split())

df_opp.head()

/tmp/ipython-input-1570504946.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_opp['description'] = df_opp['description'].apply(lambda x: x.split())
/tmp/ipython-input-1570504946.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_opp['learning_points'] = df_opp['learning_points'].apply(lambda x: x.split())


,id,title,company_name,description,backgrounds,skills,languages,study_levels,learning_points
0,1329997,Recruitment Consultant,Non Stop Consulting,"[Step, into, a, fast-paced, sales, environment...","[Sales, Marketing, Sports]","[Communication, Writing, Negotiation, Database...","[Dutch, English, French]",[High School],"[build, and, develop, a, niche, market, by, ex..."
1,1329970,Logistics Coordinator Intern,HILTI Panama,"[Hilti, is, where, innovation, is, improving, ...",[Business administration],"[SAP, Database Management, Database Administra...","[Spanish, English]",[Bachelor],"[communication, with, logistics, stakeholders:..."
2,1329930,Sx Marketing assistant LATAM GEM,Alcon Centroamérica PA,"[Provide, administrative, support, to, ensure,...",[Marketing],"[Microsoft Excel, Microsoft Powerpoint, Data A...","[Spanish, English]",[Bachelor],"[end-to-end, management, and, tracking, of, pr..."
3,1329918,"Accelerate Romania | Marketing, Social Media &...",goodStory,"[-, GoodStory, is, an, events, and, marketing,...","[Marketing, Communication & journalism, Graphi...","[Photoshop, Social Media Management, Microsoft...",[English],[Bachelor],"[-, assisting, with, photo, and, video, conten..."
4,1329895,Office Manager (EU only),Caps & Collars GmbH,"[Caps, &, Collars, GmbH, is, looking, for, a, ...","[Business administration, Tourism & hotel mana...","[Event Planning, Communication, Client servici...",[English],[Bachelor],"[make, sure, the, co-working, space, is, clean..."


In [ ]:
df_opp['tags'] = df_opp['description'] + df_opp['learning_points'] + df_opp['backgrounds'] + df_opp['skills'] + df_opp['languages'] + df_opp['study_levels']

/tmp/ipython-input-2370024102.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_opp['tags'] = df_opp['description'] + df_opp['learning_points'] + df_opp['backgrounds'] + df_opp['skills'] + df_opp['languages'] + df_opp['study_levels']


In [ ]:
new_df_opp = df_opp[['title','company_name','tags']]
new_df_opp.head()

,title,company_name,tags
0,Recruitment Consultant,Non Stop Consulting,"[Step, into, a, fast-paced, sales, environment..."
1,Logistics Coordinator Intern,HILTI Panama,"[Hilti, is, where, innovation, is, improving, ..."
2,Sx Marketing assistant LATAM GEM,Alcon Centroamérica PA,"[Provide, administrative, support, to, ensure,..."
3,"Accelerate Romania | Marketing, Social Media &...",goodStory,"[-, GoodStory, is, an, events, and, marketing,..."
4,Office Manager (EU only),Caps & Collars GmbH,"[Caps, &, Collars, GmbH, is, looking, for, a, ..."


In [ ]:
# Converting list to str
new_df_opp['tags'] = new_df_opp['tags'].apply(lambda x: " ".join(x))
new_df_opp.head()

/tmp/ipython-input-3709424229.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df_opp['tags'] = new_df_opp['tags'].apply(lambda x: " ".join(x))


,title,company_name,tags
0,Recruitment Consultant,Non Stop Consulting,Step into a fast-paced sales environment where...
1,Logistics Coordinator Intern,HILTI Panama,Hilti is where innovation is improving product...
2,Sx Marketing assistant LATAM GEM,Alcon Centroamérica PA,Provide administrative support to ensure smoot...
3,"Accelerate Romania | Marketing, Social Media &...",goodStory,- GoodStory is an events and marketing company...
4,Office Manager (EU only),Caps & Collars GmbH,Caps & Collars GmbH is looking for a fluent En...


In [ ]:
new_df_opp['tags'] = new_df_opp['tags'].apply(clean_text)
new_df_opp.head()

/tmp/ipython-input-2782836017.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df_opp['tags'] = new_df_opp['tags'].apply(clean_text)


,title,company_name,tags
0,Recruitment Consultant,Non Stop Consulting,step into a fast paced sales environment where...
1,Logistics Coordinator Intern,HILTI Panama,hilti is where innovation is improving product...
2,Sx Marketing assistant LATAM GEM,Alcon Centroamérica PA,provide administrative support to ensure smoot...
3,"Accelerate Romania | Marketing, Social Media &...",goodStory,goodstory is an events and marketing company w...
4,Office Manager (EU only),Caps & Collars GmbH,caps collars gmbh is looking for a fluent engl...


In [ ]:
import nltk
from nltk.stem import PorterStemmer

In [ ]:
ps = PorterStemmer()

In [ ]:
def stems(text):
    T = []

    for i in text.split():
        T.append(ps.stem(i))

    return " ".join(T)

In [ ]:
new_df_opp['tags'] = new_df_opp['tags'].apply(stems)

/tmp/ipython-input-556178010.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df_opp['tags'] = new_df_opp['tags'].apply(stems)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vector_opp = cv.fit_transform(new_df_opp['tags']).toarray()

In [ ]:
pip install PyMuPDF groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 11.8 MB/s eta 0:00:00


In [ ]:
import fitz
from groq import Groq

In [ ]:
def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

In [ ]:
def analyze_with_groq(text):
    client = Groq(api_key="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")

    prompt = f"""
You will receive the text of a CV.

Your job:
- Extract only the required information listed below.
- Always return the same fixed JSON structure.
- For 'education' and 'experience', return a list of objects.
- If any field is missing, leave it empty ("").
- Return ONLY valid JSON. No explanations.
- No Extra formatting with ```

The JSON structure MUST be exactly:

{{
  "name": "",
  "description": "",

  "education": [
    {{
      "degree": "",
      "specialty": ""
    }}
  ],

  "experience": [
    {{
      "role": "",
      "description": ""
    }}
  ],

  "languages": [],
  "skills": []
}}

Extraction rules:

1. Personal Info:
   - name
   - description (summarize the person in 2–3 sentences)

2. Education:
   Include ALL education entries. For each:
   - degree
   - specialty

3. Experience:
   Include ALL experience entries. For each:
   - role
   - description

4. Languages:
   - languages (list)

5. Skills:
   Include Hard and soft skills. For each:
   - skills (list)

CV content:
{text}
"""


    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content



In [ ]:
pdf_path = "/content/Med Helmi Essouaied Resume - English.pdf"

print("Extracting PDF text...")
pdf_text = extract_pdf_text(pdf_path)

print("Sending to Groq...")
result = analyze_with_groq(pdf_text)

print("\n--- Structured Output ---\n")
print(result)

Extracting PDF text...
Sending to Groq...

--- Structured Output ---

{
  "name": "Mohamed Helmi ESSOUAIED",
  "description": "An enthusiastic Computer Science graduate with a strong passion for data and business analysis, seeking an end-of-year project to apply my theoretical understanding to practical challenges. I thrive on tackling new challenges and continuously learning, especially in projects that involve developing innovative, data-driven solutions that can optimize business processes and deliver actionable insights. I have experience in programming languages and have worked on various projects, including developing web applications and working with Google Sheets API.",

  "education": [
    {
      "degree": "Bachelor in Computer Science",
      "specialty": "Software engineering and information systems"
    },
    {
      "degree": "Engineering",
      "specialty": "AI & Data Science"
    },
    {
      "degree": "CCNA-1",
      "specialty": ""
    }
  ],

  "experience": [
 

In [ ]:
cv_json = json.loads(result)

# Flatten personal info (KEEP LISTS)
flat = {
    "name": cv_json["name"],
    "description": cv_json["description"],
    "languages": cv_json["languages"],   # <-- list, no join
    "skills": cv_json["skills"],          # <-- list, no join
}


# Single-row summary DataFrame (list fields included)
df_cv = pd.DataFrame([flat])



def merge_list_of_dicts(data, keys_to_extract):
    """
    data: list of dictionaries
    keys_to_extract: list of keys to extract from each dict
    returns: merged list of clean tokens
    """
    tokens = []

    for item in data:
        for key in keys_to_extract:
            if key in item and item[key]:
                cleaned = clean_text(item[key])
                tokens.extend(cleaned.split())
    return tokens

df_cv["education"] = [merge_list_of_dicts(cv_json["education"], ["degree", "specialty"])]
df_cv["experience"] = [merge_list_of_dicts(cv_json["experience"], ["role", "description"])]
df_cv["description"] = df_cv["description"].apply(lambda x:x.split())

for col in ["languages","description","skills"]:
    if col in df_cv.columns:
        df_cv[col] = [clean_text(" ".join(df_cv[col][0]))]
    else:
        df_cv[col] = [""]

print(df_cv.head())


                      name                                        description  \
0  Mohamed Helmi ESSOUAIED  an enthusiastic computer science graduate with...   

                       languages  \
0  arabic english french spanish   

                                              skills  \
0  python java c javascript dart sql flutter flas...   

                                           education  \
0  [bachelor, in, computer, science, software, en...   

                                          experience  
0  [information, manager, involves, creating, goo...  


In [ ]:
# split description , languages, skills
df_cv["description"] = df_cv["description"].str.split()
df_cv["languages"] = df_cv["languages"].str.split()
df_cv["skills"] = df_cv["skills"].str.split()

In [ ]:
df_cv.head()

,name,description,languages,skills,education,experience
0,Mohamed Helmi ESSOUAIED,"[an, enthusiastic, computer, science, graduate...","[arabic, english, french, spanish]","[python, java, c, javascript, dart, sql, flutt...","[bachelor, in, computer, science, software, en...","[information, manager, involves, creating, goo..."


In [ ]:
df_cv["tags"] = df_cv["languages"] + df_cv["description"] +  df_cv["skills"] + df_cv["education"] + df_cv["experience"]

In [ ]:
df_cv.iloc[0]['tags']

['arabic',
 'english',
 'french',
 'spanish',
 'an',
 'enthusiastic',
 'computer',
 'science',
 'graduate',
 'with',
 'a',
 'strong',
 'passion',
 'for',
 'data',
 'and',
 'business',
 'analysis',
 'seeking',
 'an',
 'end',
 'of',
 'year',
 'project',
 'to',
 'apply',
 'my',
 'theoretical',
 'understanding',
 'to',
 'practical',
 'challenges',
 'i',
 'thrive',
 'on',
 'tackling',
 'new',
 'challenges',
 'and',
 'continuously',
 'learning',
 'especially',
 'in',
 'projects',
 'that',
 'involve',
 'developing',
 'innovative',
 'data',
 'driven',
 'solutions',
 'that',
 'can',
 'optimize',
 'business',
 'processes',
 'and',
 'deliver',
 'actionable',
 'insights',
 'i',
 'have',
 'experience',
 'in',
 'programming',
 'languages',
 'and',
 'have',
 'worked',
 'on',
 'various',
 'projects',
 'including',
 'developing',
 'web',
 'applications',
 'and',
 'working',
 'with',
 'google',
 'sheets',
 'api',
 'python',
 'java',
 'c',
 'javascript',
 'dart',
 'sql',
 'flutter',
 'flask',
 'photoshop

In [ ]:
new_df_cv = df_cv[['name','tags']]
new_df_cv.head()

,name,tags
0,Mohamed Helmi ESSOUAIED,"[arabic, english, french, spanish, an, enthusi..."


In [ ]:
new_df_cv['tags'] = new_df_cv['tags'].apply(lambda x: " ".join(x))
new_df_cv.head()

/tmp/ipython-input-392814466.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df_cv['tags'] = new_df_cv['tags'].apply(lambda x: " ".join(x))


,name,tags
0,Mohamed Helmi ESSOUAIED,arabic english french spanish an enthusiastic ...


In [ ]:
new_df_cv['tags'] = new_df_cv['tags'].apply(stems)

/tmp/ipython-input-3542884133.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df_cv['tags'] = new_df_cv['tags'].apply(stems)


In [ ]:
new_df_cv.iloc[0]['tags']

'arab english french spanish an enthusiast comput scienc graduat with a strong passion for data and busi analysi seek an end of year project to appli my theoret understand to practic challeng i thrive on tackl new challeng and continu learn especi in project that involv develop innov data driven solut that can optim busi process and deliv action insight i have experi in program languag and have work on variou project includ develop web applic and work with googl sheet api python java c javascript dart sql flutter flask photoshop figma canva googl sheet app script bachelor in comput scienc softwar engin and inform system engin ai data scienc ccna 1 inform manag involv creat googl sheet tool autom process analyz data improv workflow and contribut to data central and hadrumet dashboard creation for enhanc organiz effect backend develop nation support team inform manag thi involv work with googl sheet api to creat a backend that allow to read write updat and delet data from a central crm s

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
vector_cv = cv.transform(new_df_cv['tags']).toarray()

In [ ]:
print(vector_cv)
print(vector_opp)

[[0 0 0 ... 0 0 0]]
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [ ]:
# write a function that matches vector_cv with the closest matching vector_opp
from sklearn.metrics.pairwise import cosine_similarity
def recommend(cv_name, top_n=5):
    if cv_name not in df_cv['name'].values:
        return "CV not found."

    cv_index = df_cv[df_cv['name'] == cv_name].index[0]
    cv_vector = vector_cv[cv_index].reshape(1, -1)  # RESHAPE TO 2D

    # Compute cosine similarity
    similarity = cosine_similarity(cv_vector, vector_opp).flatten()

    # Get top N indices
    top_indices = similarity.argsort()[-top_n:][::-1]

    # Return full opportunity details
    recommendations = df.iloc[top_indices].copy()
    recommendations['match_score'] = similarity[top_indices]
    return recommendations[['id','title', 'company_name', 'location', 'description', 'match_score']].round(4)


In [ ]:
recommended = recommend("Mohamed Helmi ESSOUAIED", top_n=20)
recommended

,id,title,company_name,location,description,match_score
189,1327149,Data Engineering Intern,Altria Consulting (PVT) LTD,"Nugegoda, Sri Lanka",As an Intern - Data Engineering at Altria Cons...,0.4220
255,1325403,Electrical Engineering Intern,CGC JHANJERI MOHALI,"Sahibzada Ajit Singh Nagar, Punjab, India",\nAs an Intern - Data Engineering you will ass...,0.4084
241,1325829,Business Process Management Intern,Consistent Frozen Solutions Corporation,"Makati City, Metro Manila, Philippines","The BPM Analyst is responsible for mapping, an...",0.3947
361,1280027,Web Developer and Software in Drupart Company,Drupart Dijital Çözümler ve Tic.Ltd.Şti.,"Gebze, Türkiye","Designs and oversees software architecture, de...",0.3908
34,1329430,[Impact Brazil] - Research and Development Lab...,Nidec Global Appliance,"Joinville - Pirabeiraba, Joinville - SC, Brasil",Working in a Research and Development laborato...,0.3908
162,1327775,Accelerate Romania| Programming Intern,AQUAsoft,"Bucharest, Romania",-Collaborate with a team of developers to desi...,0.3898
210,1326503,Data Analyst,Inzeb,"Αθήνα, Ελλάδα",Data Analyst is responsible for acting as the ...,0.3871
132,1328155,Clinical Study Analyst Trainee,UCB,"Bruxelles, Belgio","Role based in HQ, Anderlecht, Brussels\nThe Cl...",0.3831
348,1296685,PHP Developer,Vitex Vietnam Software Joint Stock Company,"Đống Đa, Vietnam","As a PHP Developer, you will be responsible f...",0.3810
80,1328652,AI Engineer,DataDrill,"Novi Sad, Serbia","As an AI Engineer Intern, you will work on rea...",0.3801
